# Core Body Temperature — Spectral Analysis

Pipeline: **Data Loading → Preprocessing → PSD Computation → Visualisation**

All reusable logic lives in `src/` modules.  This notebook only calls them.

## 0 — Imports

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')

# Ensure repo root is on the path so `src.*` imports work
ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__) if '__file__' in dir() else '.', '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)

from src.loading        import load_human_csv, load_species_excel, print_species_summary, check_sampling
from src.preprocessing  import preprocess_human, clean_species_outliers
from src.psd            import compute_psd_human, compute_psd_all_species
from src.plotting       import (
    plot_timeserie, plot_species_exploration, plot_outlier_comparison,
    plot_spectral_log, plot_psd_single, plot_psd_species_overlay,
)

## 1 — Data loading

### Human data

In [ ]:
df_human = load_human_csv(r"data/group/2024-04-23T14-23-23.142Z_EB9DB7B32DCB.csv")
print(f"Shape: {df_human.shape}")
print(f"Columns: {list(df_human.columns)}")
df_human.head()

In [ ]:
plot_timeserie(df_human['datetime'], df_human['temp_C'], 'Human')

### Multi-species data

In [ ]:
species_data = load_species_excel(r"data/paper/Five_days_of_Tc_data_in_nine_species.xlsx")
df_meta = print_species_summary(species_data)

In [ ]:
check_sampling(species_data)

## 2 — Preprocessing

### Human — NaN interpolation

In [ ]:
df_human = preprocess_human(df_human, fs=1.0)
plot_timeserie(df_human['datetime'], df_human['temp_interp'], 'Human')

### Species — Exploration & outlier detection

In [ ]:
plot_species_exploration(species_data)

In [ ]:
species_data_clean = clean_species_outliers(species_data, sigma=4.0)
plot_outlier_comparison(species_data, species_data_clean)

## 3 — PSD (Welch)

### Quick Fourier view

In [ ]:
plot_spectral_log(df_human['temp_interp'], df_human['datetime'], NFFT=1024, specie='Human', log=True)

### PSD computation + log-log visualisation

In [ ]:
# Compute
freqs_human, psd_human, fs_human, nperseg_human = compute_psd_human(df_human)
psd_results = compute_psd_all_species(species_data_clean)

# Plot — Human
plot_psd_single(freqs_human, psd_human, fs_human, specie='Human', nperseg=nperseg_human)

# Plot — each species (all individuals overlaid)
for species_name, species_info in psd_results.items():
    plot_psd_species_overlay(species_info, species_name)